In [1]:
# ============================================================
# Install Required Libraries
# ============================================================

!pip install -q torch torchvision torchaudio
!pip install -q transformers datasets accelerate peft bitsandbytes
!pip install -q sentencepiece protobuf
!pip install -q pillow matplotlib seaborn pandas numpy tqdm scikit-learn
!pip install -U transformers
!pip install -U datasets
!pip install -U peft
!pip install -U trl
!pip install -U accelerate
!pip install -U bitsandbytes
!pip install -U pillow
!pip install -U sentencepiece
!pip install -U qwen-vl-utils

In [3]:
# ============================================================
# Dataset Configuration
# ============================================================

import os
from pathlib import Path

# Root folder
DATA_ROOT = Path("preprocessed_dataset")

# Train
TRAIN_IMAGE_DIR = DATA_ROOT / "train" / "images"
TRAIN_REPORT_DIR = DATA_ROOT / "train" / "reports"

# Validation
VAL_IMAGE_DIR = DATA_ROOT / "validation" / "images"
VAL_REPORT_DIR = DATA_ROOT / "validation" / "reports"

# Test
TEST_IMAGE_DIR = DATA_ROOT / "test" / "images"
TEST_REPORT_DIR = DATA_ROOT / "test" / "reports"

print(DATA_ROOT)

preprocessed_dataset


In [4]:
# ============================================================
# Verify Dataset Structure
# ============================================================

print("Train Images     :", TRAIN_IMAGE_DIR.exists())
print("Train Reports    :", TRAIN_REPORT_DIR.exists())

print("Validation Images:", VAL_IMAGE_DIR.exists())
print("Validation Reports:", VAL_REPORT_DIR.exists())

print("Test Images      :", TEST_IMAGE_DIR.exists())
print("Test Reports     :", TEST_REPORT_DIR.exists())

Train Images     : True
Train Reports    : True
Validation Images: True
Validation Reports: True
Test Images      : True
Test Reports     : True


In [5]:
# ============================================================
# Count Dataset Files
# ============================================================

print("Training Images    :", len(list(TRAIN_IMAGE_DIR.glob("*.png"))))
print("Training Reports   :", len(list(TRAIN_REPORT_DIR.glob("*.txt"))))

print("Validation Images  :", len(list(VAL_IMAGE_DIR.glob("*.png"))))
print("Validation Reports :", len(list(VAL_REPORT_DIR.glob("*.txt"))))

print("Test Images        :", len(list(TEST_IMAGE_DIR.glob("*.png"))))
print("Test Reports       :", len(list(TEST_REPORT_DIR.glob("*.txt"))))

Training Images    : 8988
Training Reports   : 9068
Validation Images  : 4594
Validation Reports : 4594
Test Images        : 4596
Test Reports       : 4596


In [6]:
# ============================================================
# Dataset Libraries
# ============================================================

import os
from PIL import Image
import matplotlib.pyplot as plt

import torch
from torch.utils.data import Dataset, DataLoader

from torchvision import transforms

In [7]:
# ============================================================
# Image Transform
# ============================================================

image_transform = transforms.Compose([

    transforms.Resize((224, 224)),

    transforms.ToTensor()

])

print("Image Transform Ready")

Image Transform Ready


In [8]:
# ============================================================
# Custom Chest X-ray Dataset
# ============================================================

class ChestXrayDataset(Dataset):

    def __init__(self, image_dir, report_dir, transform=None):

        self.image_dir = image_dir
        self.report_dir = report_dir
        self.transform = transform

        self.image_files = sorted(
            [file for file in os.listdir(image_dir) if file.endswith(".png")]
        )

        self.report_files = sorted(
            [file for file in os.listdir(report_dir) if file.endswith(".txt")]
        )

        assert len(self.image_files) == len(self.report_files), \
            "Number of images and reports do not match."

    def __len__(self):

        return len(self.image_files)

    def __getitem__(self, index):

        image_path = self.image_dir / self.image_files[index]
        report_path = self.report_dir / self.report_files[index]

        image = Image.open(image_path).convert("RGB")

        if self.transform is not None:

            image = self.transform(image)

        with open(report_path, "r", encoding="utf-8") as file:

            report = file.read().strip()

        return {

            "image": image,

            "report": report,

            "image_path": str(image_path),

            "report_path": str(report_path)

        }

print("Dataset Class Created")

Dataset Class Created


In [9]:
# ============================================================
# Dataset Objects
# ============================================================

train_dataset = ChestXrayDataset(

    TRAIN_IMAGE_DIR,

    TRAIN_REPORT_DIR,

    transform=image_transform

)

validation_dataset = ChestXrayDataset(

    VAL_IMAGE_DIR,

    VAL_REPORT_DIR,

    transform=image_transform

)

test_dataset = ChestXrayDataset(

    TEST_IMAGE_DIR,

    TEST_REPORT_DIR,

    transform=image_transform

)

print("=" * 60)

print("Training Samples   :", len(train_dataset))

print("Validation Samples :", len(validation_dataset))

print("Test Samples       :", len(test_dataset))

print("=" * 60)

AssertionError: Number of images and reports do not match.

In [10]:
import os

print("Train Images :", len(os.listdir(TRAIN_IMAGE_DIR)))
print("Train Reports:", len(os.listdir(TRAIN_REPORT_DIR)))

print()

print("Validation Images :", len(os.listdir(VAL_IMAGE_DIR)))
print("Validation Reports:", len(os.listdir(VAL_REPORT_DIR)))

print()

print("Test Images :", len(os.listdir(TEST_IMAGE_DIR)))
print("Test Reports:", len(os.listdir(TEST_REPORT_DIR)))

Train Images : 8988
Train Reports: 9068

Validation Images : 4594
Validation Reports: 4594

Test Images : 4596
Test Reports: 4596


In [11]:
# ============================================================
# Find Missing Image/Report Pairs
# ============================================================

from pathlib import Path

train_images = {
    p.stem for p in TRAIN_IMAGE_DIR.glob("*.png")
}

train_reports = {
    p.stem for p in TRAIN_REPORT_DIR.glob("*.txt")
}

missing_reports = train_images - train_reports
extra_reports = train_reports - train_images

print("Missing Reports :", len(missing_reports))
print("Extra Reports   :", len(extra_reports))

print("\nFirst 10 Extra Reports")

print(sorted(list(extra_reports))[:10])

Missing Reports : 5140
Extra Reports   : 5220

First 10 Extra Reports
['000000', '000002', '000007', '000015', '000024', '000029', '000030', '000034', '000035', '000036']


In [12]:
from pathlib import Path

images = sorted([p.name for p in TRAIN_IMAGE_DIR.glob("*.png")])

print(images[:20])

reports = sorted([p.name for p in TRAIN_REPORT_DIR.glob("*.txt")])

print(reports[:20])

['000003.png', '000004.png', '000005.png', '000008.png', '000009.png', '000012.png', '000013.png', '000014.png', '000016.png', '000017.png', '000018.png', '000019.png', '000020.png', '000025.png', '000026.png', '000027.png', '000028.png', '000031.png', '000032.png', '000037.png']
['000000.txt', '000002.txt', '000003.txt', '000007.txt', '000009.txt', '000012.txt', '000015.txt', '000016.txt', '000019.txt', '000020.txt', '000024.txt', '000025.txt', '000028.txt', '000029.txt', '000030.txt', '000031.txt', '000034.txt', '000035.txt', '000036.txt', '000037.txt']
